# 🚀 Advanced Catfish Detection ML Pipeline
**Developed by Group 7 (WIA1006)**

This notebook contains the exact, optimized training pipeline used by the Live AI Scanner. We have removed PCA compression so the Models train directly on 51 raw features, resulting in incredible accuracy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold
from imblearn.combine import SMOTETomek
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

# Import ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

print("Libraries imported successfully!")

In [ ]:
# 1. Load Dataset
# Make sure to upload 'dating_app_behavior_dataset.csv' to your Colab session!
try:
    df_raw = pd.read_csv('dating_app_behavior_dataset.csv')
    print(f"Loaded dataset with {df_raw.shape[0]} rows and {df_raw.shape[1]} columns")
except FileNotFoundError:
    print("⚠️ ERROR: Please upload 'dating_app_behavior_dataset.csv' to the Colab files pane!")
    df_raw = None

NUM_RAW_COLUMNS = ["message_sent_count", "app_usage_time_min", "swipe_right_ratio", "bio_length", "profile_pics_count", "age"]
EPS = 0.01

if df_raw is not None:
    # Basic Cleaning
    for column in NUM_RAW_COLUMNS:
        if column in df_raw.columns:
            df_raw[column] = pd.to_numeric(df_raw[column], errors="coerce")
    
    df = df_raw.dropna().reset_index(drop=True)
    
    # Outlier Filtering (z < 4)
    valid_numeric = [column for column in NUM_RAW_COLUMNS if column in df.columns]
    zscores = ((df[valid_numeric] - df[valid_numeric].mean()) / (df[valid_numeric].std(ddof=0) + EPS)).abs()
    df = df[(zscores < 4).all(axis=1)].reset_index(drop=True)
    
    print(f"Dataset after cleaning: {df.shape[0]} rows")

In [ ]:
# 2. Advanced Feature Engineering
def engineer_features(df):
    out = df.copy()
    out["engagement_score"] = out["message_sent_count"] / (out["app_usage_time_min"] + 1)
    out["swipe_msg_ratio"] = out["message_sent_count"] / (out["swipe_right_ratio"] + EPS)
    out["msg_per_minute"] = out["message_sent_count"] / (out["app_usage_time_min"] + EPS)
    out["bio_efficiency"] = out["bio_length"] / (out["message_sent_count"] + 1)
    out["bio_per_swipe"] = out["bio_length"] / (out["swipe_right_ratio"] + EPS)
    out["bio_per_minute"] = out["bio_length"] / (out["app_usage_time_min"] + 1)
    out["swipe_intensity"] = out["swipe_right_ratio"] / (out["app_usage_time_min"] + EPS)
    out["swipe_x_msg"] = out["swipe_right_ratio"] * out["message_sent_count"]

    if "profile_pics_count" in out.columns:
        out["pic_msg_ratio"] = out["profile_pics_count"] / (out["message_sent_count"] + 1)
        out["pic_swipe_ratio"] = out["profile_pics_count"] / (out["swipe_right_ratio"] + EPS)
        out["pic_per_minute"] = out["profile_pics_count"] / (out["app_usage_time_min"] + 1)

    out["Target"] = (out["match_outcome"] == "Catfished").astype(int)
    return out

if df_raw is not None:
    engineered = engineer_features(df)
    print(f"Engineered dataset features: {engineered.shape[1]}")

In [ ]:
# 3. Preparation & Scaling
if df_raw is not None:
    DROP_COLUMNS = ["match_outcome", "user_id", "Target", "location_name", "swipe_time_of_day", "app_usage_time_label", "swipe_right_label"]
    x_base = engineered.drop(columns=[col for col in DROP_COLUMNS if col in engineered.columns])
    
    # Drop categorical columns with >50 unique values
    for column in x_base.select_dtypes(include="object").columns:
        if x_base[column].nunique() > 50:
            x_base = x_base.drop(columns=[column])

    # One Hot Encode
    x_ohe = pd.get_dummies(x_base, drop_first=True).astype(float)

    # Drop Highly Correlated Features
    corr = x_ohe.corr().abs()
    corr = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    correlated_drop = [column for column in corr.columns if any(corr[column] > 0.95)]
    x_ohe = x_ohe.drop(columns=correlated_drop)

    # Variance Threshold
    selector = VarianceThreshold(threshold=0.01)
    x_values = selector.fit_transform(x_ohe)
    x = pd.DataFrame(x_values, columns=x_ohe.columns[selector.get_support()])
    y = engineered["Target"]

    # Train Test Split
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
    
    # Robust Scaling
    num_cols = x_train.select_dtypes(include=["float64", "int64"]).columns.tolist()
    scaler = RobustScaler()
    x_train_scaled = x_train.copy()
    x_test_scaled = x_test.copy()
    x_train_scaled[num_cols] = scaler.fit_transform(x_train_scaled[num_cols])
    x_test_scaled[num_cols] = scaler.transform(x_test_scaled[num_cols])
    
    x_train_arr = x_train_scaled.values.astype(np.float64)
    x_test_arr = x_test_scaled.values.astype(np.float64)
    y_train_arr = y_train.values
    y_test_arr = y_test.values
    
    print(f"Final training set shape: {x_train_arr.shape}")

In [ ]:
# 4. SMOTE-Tomek Resampling
if df_raw is not None:
    print("Balancing dataset with SMOTE-Tomek...")
    smote_tomek = SMOTETomek(random_state=42)
    train_resampled, y_train_resampled = smote_tomek.fit_resample(x_train_arr, y_train_arr)
    print(f"Resampled dataset shape: {train_resampled.shape}")

In [ ]:
# 5. Advanced Model Training (Without PCA for Maximum Feature Accuracy)
if df_raw is not None:
    positive_weight = float((y_train_resampled == 0).sum() / max((y_train_resampled == 1).sum(), 1))
    
    base_models = {
        "Logistic Regression": LogisticRegression(max_iter=3000, solver="saga", class_weight="balanced", random_state=42, n_jobs=-1),
        "Decision Tree": DecisionTreeClassifier(class_weight="balanced", max_features="sqrt", random_state=42),
        "Random Forest": RandomForestClassifier(class_weight="balanced_subsample", max_features="sqrt", random_state=42, n_jobs=-1),
        "Extra Trees": ExtraTreesClassifier(class_weight="balanced_subsample", max_features="sqrt", random_state=42, n_jobs=-1),
        "XGBoost": XGBClassifier(scale_pos_weight=positive_weight, eval_metric="auc", tree_method="hist", random_state=42, n_jobs=-1, verbosity=0),
        "MLP Neural Network": MLPClassifier(early_stopping=True, max_iter=500, random_state=42)
    }

    param_grids = {
        "Logistic Regression": {"C": [0.01, 0.1, 0.5, 1, 5], "penalty": ["l2"]},
        "Decision Tree": {"max_depth": [5, 10, 15, None], "min_samples_split": [5, 10, 20], "min_samples_leaf": [2, 4, 8]},
        "Random Forest": {"n_estimators": [100, 200, 300], "max_depth": [10, 15, 20], "min_samples_split": [2, 5, 10]},
        "Extra Trees": {"n_estimators": [100, 200, 300], "max_depth": [10, 15, 20], "min_samples_split": [2, 5, 10]},
        "XGBoost": {"n_estimators": [100, 200, 300], "learning_rate": [0.03, 0.05, 0.1], "max_depth": [4, 6, 8], "subsample": [0.8, 1.0]},
        "MLP Neural Network": {"hidden_layer_sizes": [(128, 64), (256, 128, 64)], "alpha": [0.0001, 0.001]}
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    tuned_models = {}

    print("Training and Tuning 6 Ensembled Models...")
    for name, model in base_models.items():
        print(f"Training {name}...")
        rs = RandomizedSearchCV(model, param_grids[name], n_iter=5, cv=cv, scoring="f1_macro", random_state=42, n_jobs=-1)
        rs.fit(train_resampled, y_train_resampled)
        tuned_models[name] = rs.best_estimator_
        print(f"  Best params: {rs.best_params_}")

In [ ]:
# 6. Evaluation and ROC Curves
if df_raw is not None:
    plt.figure(figsize=(10, 8))
    for name, model in tuned_models.items():
        probs = model.predict_proba(x_test_arr)[:, 1]
        fpr, tpr, _ = roc_curve(y_test_arr, probs)
        auc_score = roc_auc_score(y_test_arr, probs)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_score:.3f})")
    
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve Analysis - Advanced Models")
    plt.legend(loc="lower right")
    plt.show()